###  Anomaly Detection and Maintenance Alerts

The purpose of this analysis is to identify unusual measurements in the robot's operating data. Unusual measurements may indicate that the robot is behaving differently from its normal operating pattern and may require further inspection.

The dataset contains measurements from 14 robot axes. We first examine which axes contain usable data. We then study the measurements to understand their normal behaviour before identifying potential anomalies.

An anomaly does not necessarily mean that the robot has failed. It means that the measurement is unusually high compared with the other measurements for that axis.


In [10]:
import sys
#sys.path.append("..")

sys.path.append("../..")

from src.data_service.datacollection import StreamingSimulator
ss = StreamingSimulator("../../data/RMBR4-2_export_test.csv")

In [11]:
import pandas as pd

### Test 1: Check Available Axis Measurements

Before analyzing the robot's behaviour, we need to determine which axes contain actual measurements.

Some columns in the dataset may not contain any values. We should not use empty columns for anomaly detection because they cannot provide useful information about the robot's behaviour.

The following code checks the number of missing and available measurements for each robot axis.


In [12]:
axis_columns = [column for column in ss.data.columns if column.startswith("Axis")]

print("Missing values in each Axis:")
print(ss.data[axis_columns].isna().sum())

print("\nNumber of actual values in each Axis:")
print(ss.data[axis_columns].notna().sum())

Missing values in each Axis:
Axis #1         0
Axis #2         0
Axis #3         0
Axis #4         0
Axis #5         0
Axis #6         0
Axis #7         0
Axis #8         0
Axis #9     39672
Axis #10    39672
Axis #11    39672
Axis #12    39672
Axis #13    39672
Axis #14    39672
dtype: int64

Number of actual values in each Axis:
Axis #1     39672
Axis #2     39672
Axis #3     39672
Axis #4     39672
Axis #5     39672
Axis #6     39672
Axis #7     39672
Axis #8     39672
Axis #9         0
Axis #10        0
Axis #11        0
Axis #12        0
Axis #13        0
Axis #14        0
dtype: int64


### Test 2: Check Zero and Non-Zero Measurements

Next, we check how often each robot axis has a value of zero.

A zero measurement is not automatically considered a problem. The dataset contains many zero values, so treating every zero as an anomaly would create many false alerts.

Instead, we first understand how frequently zero and non-zero measurements occur before deciding how to identify unusual values.


In [8]:
import pandas as pd
import sys

sys.path.append("..")

from src.datacollection import StreamingSimulator

ss = StreamingSimulator("../data/RMBR4-2_export_test.csv")

analysis_data = ss.data.copy()

usable_axes = [
    "Axis #1",
    "Axis #2",
    "Axis #3",
    "Axis #4",
    "Axis #5",
    "Axis #6",
    "Axis #7",
    "Axis #8"
]

analysis_data["Time"] = pd.to_datetime(analysis_data["Time"])

print("Analysis data loaded successfully.")
print("Number of rows:", len(analysis_data))

Analysis data loaded successfully.
Number of rows: 39672


In [9]:
# Test 2: Check how many non-zero measurements each axis has

for axis in usable_axes:
    non_zero = (analysis_data[axis] > 0).sum()
    zero = (analysis_data[axis] == 0).sum()

    print(f"{axis}:")
    print(f"  Zero values: {zero}")
    print(f"  Non-zero values: {non_zero}")

Axis #1:
  Zero values: 25914
  Non-zero values: 13758
Axis #2:
  Zero values: 25822
  Non-zero values: 13850
Axis #3:
  Zero values: 25844
  Non-zero values: 13828
Axis #4:
  Zero values: 26004
  Non-zero values: 13668
Axis #5:
  Zero values: 25864
  Non-zero values: 13808
Axis #6:
  Zero values: 26073
  Non-zero values: 13599
Axis #7:
  Zero values: 26010
  Non-zero values: 13662
Axis #8:
  Zero values: 25958
  Non-zero values: 13714


### Test 3: Descriptive Statistics

We use descriptive statistics to understand the normal range of measurements for each usable robot axis.

The statistics include the average value, standard deviation, minimum, maximum, and different percentiles.

This is important because the robot axes do not all have the same measurement range. Understanding these differences helps us choose a suitable method for identifying unusual measurements.


In [10]:
# Test 3: Descriptive statistics for usable axes

print(
    analysis_data[usable_axes].describe()
)

            Axis #1       Axis #2       Axis #3       Axis #4       Axis #5  \
count  39672.000000  39672.000000  39672.000000  39672.000000  39672.000000   
mean       0.725743      3.613374      2.710336      0.620222      0.954521   
std        2.162120      6.879962      5.111901      1.574897      2.100186   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        0.000000      0.000000      0.000000      0.000000      0.000000   
50%        0.000000      0.000000      0.000000      0.000000      0.000000   
75%        0.312710      4.217190      4.586190      0.516190      0.800090   
max       23.609300     51.713230     41.855560     15.666300     20.750760   

            Axis #6       Axis #7       Axis #8  
count  39672.000000  39672.000000  39672.000000  
mean       0.599427      0.870145      0.102214  
std        1.815498      2.166811      0.423075  
min        0.000000      0.000000      0.000000  
25%        0.000000      0.000000     

### Step 4: Robot State and Maintenance Alerts

After examining the robot data, we now identify unusual measurements and determine whether they may indicate a change in the robot's operating state.

A measurement is considered a **potential anomaly** when it is higher than the 99th-percentile threshold for its individual axis. This threshold was selected because the dataset does not provide confirmed failure labels. Therefore, the analysis identifies unusual behaviour rather than confirming an equipment failure.

A single anomalous axis does not automatically mean that the robot has changed state or requires maintenance. However, when **two or more axes show anomalous measurements in the same reading**, this is treated as a stronger indication of unusual robot behaviour and generates a **potential Maintenance Notification alert**.

These alerts are intended to support human inspection and maintenance decisions. They do not confirm that a specific robot component has failed.


In [11]:
# Test 4: Detect potential anomalies

analysis_data["Anomaly"] = False

for axis in usable_axes:

    threshold = analysis_data[axis].quantile(0.99)

    print(f"{axis} threshold: {threshold}")

    analysis_data.loc[
        analysis_data[axis] > threshold,
        "Anomaly"
    ] = True

print("\nTotal records:", len(analysis_data))
print("Potential anomalies:", analysis_data["Anomaly"].sum())
print("Normal records:", (~analysis_data["Anomaly"]).sum())

Axis #1 threshold: 11.36165
Axis #2 threshold: 31.36531
Axis #3 threshold: 24.72325
Axis #4 threshold: 8.343922000000022
Axis #5 threshold: 9.885
Axis #6 threshold: 10.34957
Axis #7 threshold: 8.09655
Axis #8 threshold: 2.968319800000017

Total records: 39672
Potential anomalies: 2591
Normal records: 37081


### Anomaly Findings

The analysis identified **2,591 records containing at least one potential anomalous measurement** out of 39,672 records.

These records represent approximately **6.5% of the dataset**.

The presence of an anomaly does not by itself mean that the robot is in a failed state. It means that at least one axis has produced an unusually high measurement compared with the normal range observed for that axis.

Most records (**37,081**) did not contain an anomalous axis, indicating that the majority of the observed readings followed the expected measurement range.


### Test 5: Identify Potential Maintenance Alerts



In [12]:
# Test 5: Count anomalous axes in each record

analysis_data["Anomaly_Axis_Count"] = 0

for axis in usable_axes:

    threshold = analysis_data[axis].quantile(0.99)

    analysis_data["Anomaly_Axis_Count"] += (
        analysis_data[axis] > threshold
    ).astype(int)

print(
    analysis_data["Anomaly_Axis_Count"]
    .value_counts()
    .sort_index()
)

multi_axis_events = analysis_data[
    analysis_data["Anomaly_Axis_Count"] >= 2
].copy()

print(
    "\nPotential maintenance-alert events:",
    len(multi_axis_events)
)

Anomaly_Axis_Count
0    37081
1     2249
2      314
3       28
Name: count, dtype: int64

Potential maintenance-alert events: 342


### Maintenance Notification Alerts

* **37,081 readings:** No anomalies
* **2,249 readings:** 1 anomalous axis
* **342 readings:** 2 or more anomalous axes

We use **2+ anomalous axes in the same reading** as a potential Maintenance Notification.

**Result: 342 potential maintenance-alert events.**

These alerts indicate unusual robot behaviour and should be reviewed for possible maintenance. They are **warnings, not confirmed equipment failures**.
